In [1]:
import torch;
import random;
from PIL import Image;
import torch.nn as nn;
from torch import optim;
from os.path import join;
from typing import Tuple;
import torch.nn.functional as F;
from torch.nn.utils import spectral_norm;
from torch.utils.data import Dataset, DataLoader;
from torch.optim.lr_scheduler import CosineAnnealingLR;
from torchvision.transforms.functional import to_tensor, hflip, vflip;

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu");
batch_size = 64;
learning_rate = 1e-3;
epoch_count = 2000;

In [3]:
class SWIRDataset(Dataset):

    def __init__(self, root: str, transform: callable = None) -> None:
        super().__init__();
        self.sources = [];
        self.targets = [];
        for i in range(3):
            for idx in range(1, len(self) // 3 + 1):
                # read source
                img_800 = to_tensor(Image.open(join(root, f"Nuts_Fruits{i}", "128", "940nm", f"{idx:03}_128_940.png")).convert("L"));
                img_1050 = to_tensor(Image.open(join(root, f"Nuts_Fruits{i}", "128", "1065nm", f"{idx:03}_128_1065.png")).convert("L"));
                img_1550 = to_tensor(Image.open(join(root, f"Nuts_Fruits{i}", "128", "1550nm", f"{idx:03}_128_1550.png")).convert("L"));
                source = torch.cat([img_800, img_1050, img_1550], dim = 0);
                self.sources.append(source);
                # read target
                img_800 = to_tensor(Image.open(join(root, f"Nuts_Fruits{i}", "512", "940nm", f"{idx:03}_512_940.png")).convert("L"));
                img_1050 = to_tensor(Image.open(join(root, f"Nuts_Fruits{i}", "512", "1065nm", f"{idx:03}_512_1065.png")).convert("L"));
                img_1550 = to_tensor(Image.open(join(root, f"Nuts_Fruits{i}", "512", "1550nm", f"{idx:03}_512_1550.png")).convert("L"));
                target = torch.cat([img_800, img_1050, img_1550], dim = 0);
                self.targets.append(target);
        self.transform = transform;

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if(self.transform != None):
            return self.transform(self.sources[idx], self.targets[idx]);
        else:
            return (self.sources[idx], self.targets[idx]);

    def __len__(self) -> int:
        return 632 * 3;

In [4]:
class DMlp(nn.Module):

    def __init__(self, dim: int, growth_rate: float = 2.0) -> None:
        super().__init__();
        hidden_dim = int(dim * growth_rate);
        self.conv_0 = nn.Sequential(
            nn.Conv2d(dim, hidden_dim, 3, 1, 1, groups = dim),
            nn.Conv2d(hidden_dim, hidden_dim, 1, 1, 0)
        );
        self.act = nn.GELU();
        self.conv_1 = nn.Conv2d(hidden_dim, dim, 1, 1, 0);

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_0(x);
        x = self.act(x);
        x = self.conv_1(x);
        return x;

class PCFN(nn.Module):

    def __init__(self, dim: int, growth_rate: float = 2.0, p_rate: float = 0.25) -> None:
        super().__init__();
        hidden_dim = int(dim * growth_rate);
        p_dim = int(hidden_dim * p_rate);
        self.conv_0 = nn.Conv2d(dim, hidden_dim, 1, 1, 0);
        self.conv_1 = nn.Conv2d(p_dim, p_dim, 3, 1, 1);
        self.act = nn.GELU();
        self.conv_2 = nn.Conv2d(hidden_dim, dim, 1, 1, 0);
        self.p_dim = p_dim;
        self.hidden_dim = hidden_dim;

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if(self.training):
            x = self.act(self.conv_0(x));
            x1, x2 = torch.split(x, [self.p_dim, self.hidden_dim - self.p_dim], dim = 1);
            x1 = self.act(self.conv_1(x1));
            x = self.conv_2(torch.cat([x1, x2], dim = 1));
        else:
            x = self.act(self.conv_0(x));
            x[:, :self.p_dim, :, :] = self.act(self.conv_1(x[:, :self.p_dim, :, :]));
            x = self.conv_2(x);
        return x;

class SMFA(nn.Module):

    def __init__(self, dim: int = 36) -> None:
        super().__init__();
        self.linear_0 = nn.Conv2d(dim, dim * 2, 1, 1, 0);
        self.linear_1 = nn.Conv2d(dim, dim, 1, 1, 0);
        self.linear_2 = nn.Conv2d(dim * 2, dim, 1, 1, 0);
        self.lde = DMlp(dim, 2);
        self.dw_conv = nn.Conv2d(dim, dim, kernel_size = 3, dilation = 9, padding = 9, groups = dim);
        self.gelu = nn.GELU();
        self.alpha = nn.Parameter(torch.ones((1, dim, 1, 1)));
        self.beta = nn.Parameter(torch.zeros((1, dim, 1, 1)));

    def forward(self, f: torch.Tensor) -> torch.Tensor:
        _, _, h, w = f.shape;
        y, x = self.linear_0(f).chunk(2, dim = 1);
        x_s = self.dw_conv(F.max_pool2d(x, kernel_size = 9, padding = 4, stride = 1));
        x_v = torch.var(x, dim = (-2, -1), keepdim = True);
        x_l = x * self.gelu(self.linear_1(x_s * self.alpha + x_v * self.beta));
        y_d = self.lde(y);
        return self.linear_2(torch.cat([x_l, y_d], dim = 1));

class LayerNorm(nn.Module):

    def __init__(self, normalized_shape: int, eps: float = 1e-6) -> None:
        super().__init__();
        self.weight = nn.Parameter(torch.ones(normalized_shape));
        self.bias = nn.Parameter(torch.zeros(normalized_shape));
        self.eps = eps;

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(1, keepdim = True);
        s = (x - u).pow(2).mean(1, keepdim = True);
        x = (x - u) / torch.sqrt(s + self.eps);
        x = self.weight[:, None, None] * x + self.bias[:, None, None];
        return x;

class FMB(nn.Module):

    def __init__(self, dim: int, ffn_scale: float = 2.0) -> None:
        super().__init__();
        self.norm1 = LayerNorm(dim);
        self.smfa = SMFA(dim);
        self.norm2 = LayerNorm(dim);
        self.pcfn = PCFN(dim, ffn_scale);

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.smfa(self.norm1(x)) + x;
        x = self.pcfn(self.norm2(x)) + x;
        return x;

class SMFANet(nn.Module):

    def __init__(self, dim: int = 36, n_blocks: int = 8, ffn_scale: float = 2.0, upscaling_factor: int = 4) -> None:
        super().__init__();
        self.upscaling_factor = upscaling_factor;
        self.to_feat = nn.Conv2d(3, dim, 3, 1, 1);
        self.feats = nn.Sequential(*[FMB(dim, ffn_scale) for _ in range(n_blocks)]);
        self.to_img = nn.Sequential(
            nn.Conv2d(dim, 3 * upscaling_factor ** 2, 3, 1, 1),
            nn.PixelShuffle(upscaling_factor)
        );

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = F.interpolate(x, scale_factor = self.upscaling_factor, mode = "bicubic");
        x = self.to_feat(x);
        x = self.feats(x) + x;
        x = self.to_img(x);
        return x + residual;

In [5]:
class FFTLoss(nn.Module):

    def __init__(self, loss_weight: float = 1.0, reduction: str = 'mean') -> None:
        super().__init__();
        self.loss_weight = loss_weight;
        self.criterion = torch.nn.L1Loss(reduction = reduction);

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred_fft = torch.fft.rfft2(pred);
        target_fft = torch.fft.rfft2(target);
        pred_fft = torch.stack([pred_fft.real, pred_fft.imag], dim = -1);
        target_fft = torch.stack([target_fft.real, target_fft.imag], dim = -1);
        return self.loss_weight * self.criterion(pred_fft, target_fft);

In [6]:
def transform(source: torch.Tensor, target: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    y, x = random.randint(0, source.size(1) - 64 - 1), random.randint(0, source.size(2) - 64 - 1);
    source = source[:, y:(y + 64), x:(x + 64)];
    target = target[:, (y * 4):(y * 4 + 256), (x * 4):(x * 4 + 256)];
    if(random.random() <= 0.5):
        source = vflip(source);
        target = vflip(target);
    if(random.random() <= 0.5):
        source = hflip(source);
        target = hflip(target);
    return (source, target);

**Pretrain generator**

In [7]:
g = SMFANet().train().to(device);
g_opt = optim.Adam(g.parameters(), lr = learning_rate);

dataset = SWIRDataset("/kaggle/input/dataset-dec", transform);
data_loader = DataLoader(dataset, batch_size = batch_size, shuffle = True, num_workers = 2);

scheduler = CosineAnnealingLR(g_opt, T_max = epoch_count * len(data_loader), eta_min = 1e-6);

fft = FFTLoss(0.1);

avg_loss = 0.0;
for epoch in range(epoch_count):
    for (source, target) in data_loader:
        source = source.to(device);
        target = target.to(device);
        pred = g(source);
        loss = F.l1_loss(pred, target) + fft(pred, target);
        g_opt.zero_grad();
        loss.backward();
        g_opt.step();
        scheduler.step();
        avg_loss += loss.item();
    if((epoch + 1) % 20 == 0):
        print(f"epoch: {epoch + 1}, loss: {avg_loss / (len(data_loader) * 20)}");
        avg_loss = 0.0;
        torch.save(g.state_dict(), "smfanet.pth");

epoch: 20, loss: 0.6784925881028175
epoch: 40, loss: 0.618751516242822
epoch: 60, loss: 0.5980814841389656
epoch: 80, loss: 0.5886339938640595
epoch: 100, loss: 0.5826867419481278
epoch: 120, loss: 0.5788869149982929
epoch: 140, loss: 0.575748215218385
epoch: 160, loss: 0.5747339902818203
epoch: 180, loss: 0.5709081016977628
epoch: 200, loss: 0.5707686501741409
epoch: 220, loss: 0.5683205274740855
epoch: 240, loss: 0.566954844246308
epoch: 260, loss: 0.5652301252881686
epoch: 280, loss: 0.5644957837462425
epoch: 300, loss: 0.5624656840165456
epoch: 320, loss: 0.5611825135350227
epoch: 340, loss: 0.560221769263347
epoch: 360, loss: 0.5593504236141841
epoch: 380, loss: 0.5586728995045026
epoch: 400, loss: 0.5577615609765053
epoch: 420, loss: 0.556888118882974
epoch: 440, loss: 0.5563854042689006
epoch: 460, loss: 0.5557564998169741
epoch: 480, loss: 0.5552603719135125
epoch: 500, loss: 0.5541649190088113
epoch: 520, loss: 0.5536338058114052
epoch: 540, loss: 0.5526785651346048
epoch: 560

In [7]:
class Discriminator(nn.Module):

    def __init__(self, num_in_ch: int = 3, num_feat: int = 32, skip_connection: bool = True) -> None:
        super().__init__();
        self.skip_connection = skip_connection;
        # the first convolution
        self.conv0 = nn.Conv2d(num_in_ch, num_feat, kernel_size = 3, stride = 1, padding = 1);
        # downsample
        self.conv1 = spectral_norm(nn.Conv2d(num_feat, num_feat * 2, 4, 2, 1, bias = False));
        self.conv2 = spectral_norm(nn.Conv2d(num_feat * 2, num_feat * 4, 4, 2, 1, bias = False));
        self.conv3 = spectral_norm(nn.Conv2d(num_feat * 4, num_feat * 8, 4, 2, 1, bias = False));
        # upsample
        self.conv4 = spectral_norm(nn.Conv2d(num_feat * 8, num_feat * 4, 3, 1, 1, bias = False));
        self.conv5 = spectral_norm(nn.Conv2d(num_feat * 4, num_feat * 2, 3, 1, 1, bias = False));
        self.conv6 = spectral_norm(nn.Conv2d(num_feat * 2, num_feat, 3, 1, 1, bias = False));
        # extra convolutions
        self.conv7 = spectral_norm(nn.Conv2d(num_feat, num_feat, 3, 1, 1, bias = False));
        self.conv8 = spectral_norm(nn.Conv2d(num_feat, num_feat, 3, 1, 1, bias = False));
        self.conv9 = nn.Conv2d(num_feat, 1, 3, 1, 1);

    def forward(self, x: torch.Tensor) -> Tuple[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor], torch.Tensor]:
        # downsample
        x0 = F.leaky_relu(self.conv0(x), negative_slope = 0.2, inplace = True);
        x1 = F.leaky_relu(self.conv1(x0), negative_slope = 0.2, inplace = True);
        x2 = F.leaky_relu(self.conv2(x1), negative_slope = 0.2, inplace = True);
        x3 = F.leaky_relu(self.conv3(x2), negative_slope = 0.2, inplace = True);
        # upsample
        x3_ = F.interpolate(x3, scale_factor = 2, mode = 'bilinear', align_corners = False);
        x4 = F.leaky_relu(self.conv4(x3_), negative_slope = 0.2, inplace = True);
        if(self.skip_connection):
            x4 = x4 + x2;
        x4 = F.interpolate(x4, scale_factor = 2, mode = 'bilinear', align_corners = False);
        x5 = F.leaky_relu(self.conv5(x4), negative_slope = 0.2, inplace = True);
        if(self.skip_connection):
            x5 = x5 + x1;
        x5 = F.interpolate(x5, scale_factor = 2, mode = 'bilinear', align_corners = False);
        x6 = F.leaky_relu(self.conv6(x5), negative_slope = 0.2, inplace = True);
        if(self.skip_connection):
            x6 = x6 + x0;
        # extra convolutions
        out = F.leaky_relu(self.conv7(x6), negative_slope = 0.2, inplace = True);
        out = F.leaky_relu(self.conv8(out), negative_slope = 0.2, inplace = True);
        out = self.conv9(out);
        return (x0, x1, x2, x3), out;

In [8]:
def transform(source: torch.Tensor, target: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    y, x = random.randint(0, source.size(1) - 64 - 1), random.randint(0, source.size(2) - 64 - 1);
    source = source[:, y:(y + 64), x:(x + 64)];
    target = target[:, (y * 4):(y * 4 + 256), (x * 4):(x * 4 + 256)];
    if(random.random() <= 0.5):
        source = vflip(source);
        target = vflip(target);
    if(random.random() <= 0.5):
        source = hflip(source);
        target = hflip(target);
    return (source, target);

**We need some initial pretraining for discriminator...**

In [9]:
epoch_count = 500;

g = SMFANet().eval().to(device);
g.load_state_dict(torch.load("/kaggle/input/smfanet-v3/pytorch/default/1/smfanet.pth"));
d = Discriminator().train().to(device);
d_opt = optim.Adam(d.parameters(), lr = 1e-4, betas = (0.9, 0.99));

dataset = SWIRDataset("/kaggle/input/dataset-dec", transform);
data_loader = DataLoader(dataset, batch_size = 16, shuffle = True, num_workers = 2);

scheduler_d = CosineAnnealingLR(d_opt, T_max = epoch_count * len(data_loader), eta_min = 1e-5);

avg_dis_loss = 0.0;
for epoch in range(epoch_count):
    for (source, target) in data_loader:
        source = source.to(device);
        target = target.to(device);
        with torch.no_grad():
            fake = g(source);
        # Train discriminator
        df_real, d_real = d(target);
        df_fake, d_fake = d(fake);
        dis_loss = F.binary_cross_entropy_with_logits(d_real, torch.ones_like(d_real)) +\
                   F.binary_cross_entropy_with_logits(d_fake, torch.zeros_like(d_fake));
        d_opt.zero_grad();
        dis_loss.backward();
        d_opt.step();
        scheduler_d.step();
        avg_dis_loss += dis_loss.item();
    if((epoch + 1) % 20 == 0):
        print(f"epoch: {epoch + 1}, dis_loss: {avg_dis_loss / (len(data_loader) * 20)}");
        avg_dis_loss = 0.0;
        torch.save(d.state_dict(), "discriminator.pth");

epoch: 20, dis_loss: 0.7676277290497507
epoch: 40, dis_loss: 0.5904798949904302
epoch: 60, dis_loss: 0.3352542131888766
epoch: 80, dis_loss: 0.15774419280708213
epoch: 100, dis_loss: 0.08485844179267893
epoch: 120, dis_loss: 0.04943009758282046
epoch: 140, dis_loss: 0.03105642083961861
epoch: 160, dis_loss: 0.020545581829031743
epoch: 180, dis_loss: 0.0138946779805261
epoch: 200, dis_loss: 0.01046244278117259
epoch: 220, dis_loss: 0.006698914492593789
epoch: 240, dis_loss: 0.005349091600926061
epoch: 260, dis_loss: 0.003953437096831347
epoch: 280, dis_loss: 0.0027874805442861697
epoch: 300, dis_loss: 0.0021778643154490252
epoch: 320, dis_loss: 0.0017395777168920695
epoch: 340, dis_loss: 0.0013605231501625757
epoch: 360, dis_loss: 0.0010194668520853156
epoch: 380, dis_loss: 0.0007710182193125953
epoch: 400, dis_loss: 0.0005672728024448986
epoch: 420, dis_loss: 0.0004971681693049726
epoch: 440, dis_loss: 0.0004110056895076908
epoch: 460, dis_loss: 0.00034046830507592684
epoch: 480, dis_l

In [ ]:
epoch_count = 1000;

g = SMFANet().train().to(device);
g.load_state_dict(torch.load("/kaggle/input/smfanet-v3/pytorch/default/1/smfanet.pth"));
g_opt = optim.Adam(g.parameters(), lr = 1e-4, betas = (0.9, 0.99));
d = Discriminator().train().to(device);
d.load_state_dict(torch.load("/kaggle/input/discriminator-v3/pytorch/default/1/discriminator.pth"));
d_opt = optim.Adam(d.parameters(), lr = 1e-4, betas = (0.9, 0.99));

dataset = SWIRDataset("/kaggle/input/dataset-dec", transform);
data_loader = DataLoader(dataset, batch_size = 16, shuffle = True, num_workers = 2);

scheduler_g = CosineAnnealingLR(g_opt, T_max = epoch_count * len(data_loader), eta_min = 2e-5);
scheduler_d = CosineAnnealingLR(d_opt, T_max = epoch_count * len(data_loader), eta_min = 2e-5);

fft = FFTLoss(0.1);

avg_l1_loss = 0.0;
avg_fft_loss = 0.0;
avg_perceptual_loss = 0.0;
avg_adv_loss = 0.0;
avg_dis_loss = 0.0;
for epoch in range(epoch_count):
    for (source, target) in data_loader:
        source = source.to(device);
        target = target.to(device);
        fake = g(source);
        # Train discriminator
        d.train();
        g.eval();
        df_real, d_real = d(target);
        df_fake, d_fake = d(fake.detach());
        dis_loss = F.binary_cross_entropy_with_logits(d_real, torch.ones_like(d_real)) +\
                   F.binary_cross_entropy_with_logits(d_fake, torch.zeros_like(d_fake));
        d_opt.zero_grad();
        dis_loss.backward();
        d_opt.step();
        scheduler_d.step();
        avg_dis_loss += dis_loss.item();
        # Train generator
        g.train();
        d.eval();
        l1_loss = F.l1_loss(fake, target);
        fft_loss = fft(fake, target);
        df_real, d_real = d(target);
        df_fake, d_fake = d(fake);
        perceptual_loss = F.l1_loss(df_fake[0], df_real[0]) * 1.0 +\
                          F.l1_loss(df_fake[1], df_real[1]) * 1.0 +\
                          F.l1_loss(df_fake[2], df_real[2]) * 1.0 +\
                          F.l1_loss(df_fake[3], df_real[3]) * 1.0;
        adv_loss = F.binary_cross_entropy_with_logits(d_fake, torch.ones_like(d_fake)) * 1e-2;
        loss = l1_loss + fft_loss + perceptual_loss + adv_loss;
        g_opt.zero_grad();
        loss.backward();
        g_opt.step();
        scheduler_g.step();
        avg_l1_loss += l1_loss.item();
        avg_fft_loss += fft_loss.item();
        avg_perceptual_loss += perceptual_loss.item();
        avg_adv_loss += adv_loss.item();
    if((epoch + 1) % 10 == 0):
        print(f"epoch: {epoch + 1}, dis_loss: {avg_dis_loss / (len(data_loader) * 10)}, l1_loss: {avg_l1_loss / (len(data_loader) * 10)}, fft_loss: {avg_fft_loss / (len(data_loader) * 10)}, perceptual_loss: {avg_perceptual_loss / (len(data_loader) * 10)}, adv_loss: {avg_adv_loss / (len(data_loader) * 10)}");
        avg_l1_loss = 0.0;
        avg_fft_loss = 0.0;
        avg_perceptual_loss = 0.0;
        avg_adv_loss = 0.0;
        avg_dis_loss = 0.0;
        torch.save(g.state_dict(), f"generator_{epoch + 1}.pth");
        torch.save(d.state_dict(), f"discriminator_{epoch + 1}.pth");

epoch: 10, dis_loss: 0.9414419492873208, l1_loss: 0.031153662319631636, fft_loss: 0.5094965890676033, perceptual_loss: 0.036195504809377575, adv_loss: 0.015625885725334413
epoch: 20, dis_loss: 0.7128135683406301, l1_loss: 0.03140550792812049, fft_loss: 0.5100988329208198, perceptual_loss: 0.0364416689729365, adv_loss: 0.017334198759195684
epoch: 30, dis_loss: 0.6549871822114752, l1_loss: 0.0318343158654806, fft_loss: 0.5120755164062276, perceptual_loss: 0.03617850362847583, adv_loss: 0.018833090511283703
epoch: 40, dis_loss: 0.6331627518940373, l1_loss: 0.031946070805317216, fft_loss: 0.5117976017609364, perceptual_loss: 0.03578036202102148, adv_loss: 0.019697130905041674
epoch: 50, dis_loss: 0.6039015094773108, l1_loss: 0.032052450508129694, fft_loss: 0.5125566155720158, perceptual_loss: 0.035167881456681156, adv_loss: 0.020659761293595577
epoch: 60, dis_loss: 0.5839121493221331, l1_loss: 0.03196648092736967, fft_loss: 0.5135779869406163, perceptual_loss: 0.03424508483501292, adv_loss